# Verify the `delta_lakehouse` skill against a real Fabric Lakehouse

Everything in this skill was measured against Delta tables built locally with
`delta-rs`. That covers the Delta protocol, but not Fabric: not a real mount,
not `abfss://`, and not tables written by Spark (which produces genuine
deletion vector files, V-Order, and its own file layout).

This notebook closes that gap. It installs the branch, pulls the skill's own
code out of the installed Markdown, and runs each claim against your table,
reporting PASS / FAIL / SKIP per claim.

**This notebook is read-only.** It never writes, deletes, compacts, or vacuums
anything. The only write is an optional summary printed to the cell output.

Branch under test: `feat/delta-lakehouse-skill`

## 1. Install

`fabric-rlm-core` is a private repo, so a plain `pip install git+https://...`
will not authenticate from a Fabric notebook.

**Option A (recommended, no token in the notebook).** Build a wheel on your
machine and upload it to the Lakehouse:

```bash
git checkout feat/delta-lakehouse-skill
python -m build --wheel
```

Upload `dist/fabric_rlm-*.whl` to `Files/wheels/` in your Lakehouse, then run
the install cell below as written.

**Option B (quick).** Use a PAT inline. Prefer Option A if this notebook will
be saved or shared, since the token lands in the notebook file.

In [ ]:
# Option A - install the wheel you uploaded to Files/wheels/ (recommended)
import glob
whl = sorted(glob.glob("/lakehouse/default/Files/wheels/fabric_rlm-*.whl"))
assert whl, "No wheel found. Upload dist/fabric_rlm-*.whl to Files/wheels/ first."
print("installing", whl[-1])
%pip install --quiet "{whl[-1]}"

# Option B - install straight from the branch with a PAT. Uncomment to use.
# PAT = ""  # a fine-grained token with read access to the repo
# %pip install --quiet "git+https://{PAT}@github.com/pawarbi/fabric-rlm-core.git@feat/delta-lakehouse-skill"

Fabric may need a session restart before the new version is importable. If the
version below is not the one you just installed, restart the session and re-run
from here (skip the install cell).

In [ ]:
import fabric_rlm
from fabric_rlm.skill_loader import SkillLoader

print("fabric_rlm", fabric_rlm.__version__, "from", fabric_rlm.__file__)
print("skills packaged:", sorted(SkillLoader().list_skills()))
assert "delta_lakehouse" in SkillLoader().list_skills(), \
    "delta_lakehouse is missing - you are on an older build, restart the session"

## 2. Point this at your table

Set these to a table you can read. A **partitioned** table with some delete or
merge history exercises the most claims; a plain table still checks the core
path. Nothing here modifies the table.

In [ ]:
WORKSPACE = "<your workspace name>"      # the workspace the lakehouse lives in
LAKEHOUSE = "<your lakehouse name>"      # without the .Lakehouse suffix
SCHEMA    = None                          # e.g. "dbo" for a schema-enabled lakehouse, else None
TABLE     = "<your table name>"

TEST_MOUNTED = True     # requires the lakehouse to be attached to this notebook
TEST_ABFSS   = True     # works whether or not it is attached

_rel = f"{SCHEMA}/{TABLE}" if SCHEMA else TABLE
MOUNTED_PATH = f"/lakehouse/default/Tables/{_rel}"
ABFSS_PATH = (
    f"abfss://{WORKSPACE}@onelake.dfs.fabric.microsoft.com/"
    f"{LAKEHOUSE}.Lakehouse/Tables/{_rel}"
)
print("mounted:", MOUNTED_PATH)
print("abfss  :", ABFSS_PATH)

## 3. Harness

`check` records one result per claim. A claim that cannot be evaluated on your
table (no tombstones, not partitioned, no deletion vectors) reports **SKIP**
with the reason rather than a misleading pass.

In [ ]:
RESULTS = []

def check(name, fn):
    """fn returns (True|False|None, note). None means 'not applicable here'."""
    try:
        ok, note = fn()
    except Exception as e:
        RESULTS.append((name, "FAIL", f"{type(e).__name__}: {str(e)[:200]}"))
        print(f"FAIL  {name}\n      {type(e).__name__}: {str(e)[:200]}")
        return
    status = {True: "PASS", False: "FAIL", None: "SKIP"}[ok]
    RESULTS.append((name, status, note))
    print(f"{status}  {name}" + (f"\n      {note}" if note else ""))

## 4. Does the router send a Fabric-shaped question to the skill?

These are the exact question shapes a user would type.

In [ ]:
from fabric_rlm.skill_router import SkillRouter

router = SkillRouter.from_loader(SkillLoader())

def _routes(q):
    d = router.route(q)
    return ("delta_lakehouse" in d.active, f"active={d.active} scores={d.scores}")

for q in [
    f"what are the top 10 rows by value in the {TABLE} delta table",
    f"explore {MOUNTED_PATH} and summarize it",
    f"profile {ABFSS_PATH}",
]:
    check(f"routes: {q[:58]}...", lambda q=q: _routes(q))

# And must NOT fire on unrelated work.
check("does not route a plain csv question", lambda: (
    "delta_lakehouse" not in router.route("count rows in Files/data.csv").active, ""))

## 5. Load the skill's own code out of the installed Markdown

This is the point of the exercise: the functions below are not retyped here,
they are extracted from the skill text that shipped in the wheel. If the skill
is wrong, this cell fails.

In [ ]:
import re

SKILL = SkillLoader().load("delta_lakehouse").content
blocks = re.findall(r"```python\n(.*?)\n```", SKILL, re.DOTALL)

resolver = [b for b in blocks if "def open_delta" in b]
discovery = [b for b in blocks if "=== profile ===" in b]
assert len(resolver) == 1, f"expected 1 open_delta block, found {len(resolver)}"
assert len(discovery) == 1, f"expected 1 discovery block, found {len(discovery)}"

exec(compile(resolver[0], "<delta_lakehouse:open_delta>", "exec"), globals())
DISCOVERY_SRC = discovery[0]
print("loaded open_delta / delta_opts / _storage_token from the installed skill")
print("discovery block:", len(DISCOVERY_SRC.splitlines()), "lines")

## 6. Can the skill open your table, both ways?

In [ ]:
import duckdb

HANDLES = {}

def _open(path, label):
    con, T = open_delta(path)
    HANDLES[label] = (con, T)
    n = con.sql(f"SELECT count(*) FROM {T}").fetchone()[0]
    engine = "delta_scan" if "delta_scan" in T else "delta-rs fallback"
    return (n >= 0, f"{engine}, {n:,} rows")

if TEST_MOUNTED:
    check("open attached lakehouse mount", lambda: _open(MOUNTED_PATH, "mount"))
else:
    check("open attached lakehouse mount", lambda: (None, "TEST_MOUNTED is False"))

if TEST_ABFSS:
    check("open abfss:// OneLake path", lambda: _open(ABFSS_PATH, "abfss"))
else:
    check("open abfss:// OneLake path", lambda: (None, "TEST_ABFSS is False"))

# Both paths must agree on the row count.
def _agree():
    if "mount" not in HANDLES or "abfss" not in HANDLES:
        return (None, "need both paths open")
    a = HANDLES["mount"][0].sql(f"SELECT count(*) FROM {HANDLES['mount'][1]}").fetchone()[0]
    b = HANDLES["abfss"][0].sql(f"SELECT count(*) FROM {HANDLES['abfss'][1]}").fetchone()[0]
    return (a == b, f"mount={a:,} abfss={b:,}")

check("mount and abfss return the same count", _agree)

PATH = MOUNTED_PATH if "mount" in HANDLES else ABFSS_PATH
con, T = HANDLES.get("mount") or HANDLES.get("abfss")

## 7. The discovery block, run verbatim

Schema, partitions, sample rows, and a per-column profile. Watch the output
size: the skill claims this is cheap enough to be mandatory on turn 1.

In [ ]:
import io, contextlib

buf = io.StringIO()
_ns = dict(globals()); _ns["path"] = PATH
with contextlib.redirect_stdout(buf):
    exec(compile(DISCOVERY_SRC, "<delta_lakehouse:discovery>", "exec"), _ns)

out = buf.getvalue()
print(out)

check("discovery prints all four sections", lambda: (
    all(s in out for s in ("=== table ===", "=== schema ===", "=== sample ===", "=== profile ===")),
    "",
))
check("discovery output stays small enough for turn 1", lambda: (
    len(out) < 4000, f"{len(out)} chars, {out.count(chr(10))} lines"))

## 8. The correctness claims

The headline one first: on a table with delete/merge history and no vacuum
since, reading the parquet directly must disagree with the Delta reader. If
your table has never been mutated there is nothing to disagree about, and the
check reports SKIP.

In [ ]:
from deltalake import DeltaTable
import pyarrow as pa, glob as _glob, os

dt = DeltaTable(PATH, storage_options=delta_opts(PATH))
live = dt.file_uris()
features = dt.protocol().reader_features or []
parts = dt.partitions()
def _dv_entries(t):
    # How many data files actually carry a deletion vector, not just the flag.
    try:
        return pa.table(t.deletion_vectors().read_all()).num_rows
    except Exception:
        return 0

DV = _dv_entries(dt)
print(f"version={dt.version()}  live files={len(live)}  reader_features={features}  "
      f"partitions={len(parts)}  deletion_vectors={DV}")

def _glob_diverges():
    if "://" in PATH:
        return (None, "physical file listing needs a filesystem path, not abfss")
    physical = _glob.glob(os.path.join(PATH, "**", "*.parquet"), recursive=True)
    if len(physical) <= len(live):
        return (None, f"no tombstoned files present ({len(physical)} on disk, {len(live)} live) - "
                      "table has no un-vacuumed delete/merge history")
    truth = con.sql(f"SELECT count(*) FROM {T}").fetchone()[0]
    wrong = con.sql(
        f"SELECT count(*) FROM read_parquet('{PATH}/**/*.parquet')").fetchone()[0]
    return (wrong != truth,
            f"glob={wrong:,} vs delta={truth:,} across {len(physical)} files on disk / {len(live)} live")

check("read_parquet glob disagrees with the Delta reader", _glob_diverges)


def _file_uris_diverges():
    if "deletionVectors" not in features:
        return (None, "table has no deletionVectors reader feature")
    if DV == 0:
        return (None, "deletionVectors is enabled but no data file carries one yet, "
                      "so there is nothing here for a parquet read to get wrong")
    truth = con.sql(f"SELECT count(*) FROM {T}").fetchone()[0]
    lst = ", ".join(f"'{u}'" for u in live)
    viauris = con.sql(f"SELECT count(*) FROM read_parquet([{lst}])").fetchone()[0]
    return (viauris != truth,
            f"read_parquet(file_uris())={viauris:,} vs delta={truth:,} - "
            "this is the 'looks rigorous but is wrong' case")

check("read_parquet(file_uris()) is wrong under deletion vectors", _file_uris_diverges)


def _delta_rs_refuses():
    if "deletionVectors" not in features:
        return (None, "table has no deletionVectors reader feature")
    from deltalake.exceptions import DeltaProtocolError
    try:
        dt.to_pyarrow_dataset()
        return (False, "to_pyarrow_dataset() succeeded - deltalake may now support DVs, "
                       "in which case the skill's hard-fail branch can be relaxed")
    except DeltaProtocolError as e:
        return (True, f"raised as expected: {str(e)[:110]}")

check("delta-rs refuses deletion-vector tables", _delta_rs_refuses)


def _num_records_upper_bound():
    adds = pa.table(dt.get_add_actions()).to_pydict()
    meta = sum(adds.get("num_records", []))
    actual = con.sql(f"SELECT count(*) FROM {T}").fetchone()[0]
    if DV == 0:
        return (meta == actual,
                f"no deletion vectors present, so metadata should be exact: {meta:,} vs {actual:,}")
    # The case only reasoning covered locally: delta-rs rewrote files rather than
    # emitting a deletion vector, so no local table ever exercised the over-count.
    return (meta >= actual,
            f"metadata={meta:,} actual={actual:,} across {DV} deletion vector(s) - "
            f"{'over-counts as predicted' if meta > actual else 'equal despite DVs'}")

check("num_records is exact without DVs, an upper bound with them", _num_records_upper_bound)


def _limit_is_biased():
    if len(parts) < 2:
        return (None, f"table has {len(parts)} partition(s); need 2+ to show the bias")
    col = list(parts[0].keys())[0]
    lim = {r[0] for r in con.sql(f"SELECT {col} FROM {T} LIMIT 20").fetchall()}
    smp = {r[0] for r in con.sql(f"SELECT {col} FROM {T} USING SAMPLE 200 ROWS").fetchall()}
    return (len(lim) < len(smp),
            f"LIMIT saw {len(lim)} distinct {col}, USING SAMPLE saw {len(smp)}")

check("LIMIT is biased to one partition, USING SAMPLE is not", _limit_is_biased)


def _approx_unique_is_approx():
    rel = con.sql(f"SUMMARIZE SELECT * FROM {T}")
    cols = [d[0] for d in rel.description]
    rows = [dict(zip(cols, r)) for r in rel.fetchall()]
    target = max(rows, key=lambda d: int(d["approx_unique"] or 0))
    name = target["column_name"]
    exact = con.sql(f'SELECT count(DISTINCT "{name}") FROM {T}').fetchone()[0]
    approx = int(target["approx_unique"])
    return (True, f"{name}: approx={approx:,} exact={exact:,} "
                  f"({'differs - quote the exact one' if approx != exact else 'equal here, still an estimate'})")

check("approx_unique is an estimate, not a fact", _approx_unique_is_approx)


def _show_encoding():
    try:
        con.sql(f"SELECT * FROM {T} LIMIT 2").show()
        return (None, "no UnicodeEncodeError on this runtime - the trap is cp1252/Windows "
                      "specific, and fetchall() is portable either way")
    except UnicodeEncodeError as e:
        return (True, f"reproduced: {str(e)[:110]}")

check(".show() encoding trap", _show_encoding)

## 9. Read-only confirmation

The table must be exactly where it started. If this fails, something in the run
wrote to your table and that is a bug worth reporting.

In [ ]:
_before = dt.version()
_after = DeltaTable(PATH, storage_options=delta_opts(PATH)).version()
check("table version unchanged by this notebook",
      lambda: (_before == _after, f"version {_before} -> {_after}"))

## 10. Summary

Paste this output back if anything failed.

In [ ]:
import collections, platform

tally = collections.Counter(s for _, s, _ in RESULTS)
print(f"fabric_rlm {fabric_rlm.__version__} | duckdb {duckdb.__version__} | "
      f"python {platform.python_version()}")
print(f"table: {PATH}")
print(f"reader_features={features} partitions={len(parts)} version={dt.version()}")
print()
for name, status, note in RESULTS:
    print(f"{status:5} {name}")
    if note:
        print(f"      {note}")
print()
print("  ".join(f"{k}={v}" for k, v in sorted(tally.items())))
if tally.get("FAIL"):
    print("\nFAILURES above are real gaps in the skill, not notebook problems.")